# GPU Tabular Transformer for Bank Marketing

This Colab-ready notebook trains a transformer-style model for the provided bank marketing data. It uses PyTorch and automatically uses a GPU when Colab has one enabled.

Before running:

1. In Colab, choose `Runtime > Change runtime type > GPU`.
2. Upload `training_data.csv` and `testing_data.csv`, or place them under `data/` in the notebook runtime.
3. Run all cells.

The notebook drops `duration` because it is post-call target leakage.

In [1]:
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

SEED = 580
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA H100 80GB HBM3


In [2]:
def find_csv(name):
    candidates = [
        Path(name),
        Path("data") / name,
        Path("/content") / name,
        Path("/content/data") / name,
        Path("/content/drive/MyDrive") / name,
        Path("/content/drive/MyDrive/data") / name,
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


train_path = find_csv("training_data.csv")
test_path = find_csv("testing_data.csv")

if train_path is None or test_path is None:
    try:
        from google.colab import files

        print("Upload training_data.csv and testing_data.csv")
        uploaded = files.upload()
        train_path = find_csv("training_data.csv")
        test_path = find_csv("testing_data.csv")
    except Exception as exc:
        raise FileNotFoundError(
            "Could not find training_data.csv and testing_data.csv. "
            "Upload them to Colab or place them under data/."
        ) from exc

print("Training CSV:", train_path)
print("Testing CSV:", test_path)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df["y"].value_counts(normalize=True).rename("class_rate"))

Upload training_data.csv and testing_data.csv


Saving testing_data.csv to testing_data.csv
Saving training_data.csv to training_data.csv
Training CSV: training_data.csv
Testing CSV: testing_data.csv
Train shape: (32951, 21)
Test shape: (8237, 21)
y
no     0.887348
yes    0.112652
Name: class_rate, dtype: float64


In [3]:
def safe_div(numerator, denominator):
    return np.nan if denominator == 0 else numerator / denominator


def metric_row(y_true, prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob)
    pred = (prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    accuracy = safe_div(tp + tn, len(y_true))
    sensitivity = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    precision = safe_div(tp, tp + fp)
    f1 = safe_div(2 * precision * sensitivity, precision + sensitivity)
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    if min(accuracy, sensitivity, specificity) <= 0:
        harmonic = 0.0
    else:
        harmonic = 3.0 / ((1.0 / accuracy) + (1.0 / sensitivity) + (1.0 / specificity))
    business_score = 0.4 * accuracy + 0.4 * sensitivity + 0.2 * specificity
    return {
        "threshold": threshold,
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy,
        "harmonic_acc_sens_spec": harmonic,
        "business_score": business_score,
        "precision": precision,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def threshold_table(y_true, prob):
    rows = [metric_row(y_true, prob, th) for th in np.r_[np.arange(0.01, 0.991, 0.001), 0.5]]
    out = pd.DataFrame(rows).drop_duplicates("threshold")
    out["auc"] = roc_auc_score(y_true, prob)
    return out


def pick_threshold(y_true, prob, objective="harmonic_acc_sens_spec"):
    tab = threshold_table(y_true, prob)
    return tab.sort_values(
        [objective, "balanced_accuracy", "accuracy"],
        ascending=[False, False, False],
    ).iloc[0]


def evaluate_points(model_name, y_true, prob, threshold_source_y, threshold_source_prob):
    rows = []
    for name, th in [
        ("default_0.50", 0.5),
        ("accuracy_threshold", pick_threshold(threshold_source_y, threshold_source_prob, "accuracy")["threshold"]),
        ("harmonic_threshold", pick_threshold(threshold_source_y, threshold_source_prob, "harmonic_acc_sens_spec")["threshold"]),
        ("business_threshold", pick_threshold(threshold_source_y, threshold_source_prob, "business_score")["threshold"]),
    ]:
        row = metric_row(y_true, prob, float(th))
        row["model"] = model_name
        row["operating_point"] = name
        row["auc"] = roc_auc_score(y_true, prob)
        rows.append(row)
    return pd.DataFrame(rows)

In [4]:
def education_order(values):
    mapping = {
        "illiterate": 0,
        "basic.4y": 1,
        "basic.6y": 2,
        "basic.9y": 3,
        "high.school": 4,
        "professional.course": 5,
        "university.degree": 6,
        "unknown": np.nan,
    }
    out = pd.Series(values).map(mapping).astype(float)
    return out.fillna(out.median()).to_numpy()


def add_features(df, campaign_cap=None):
    out = df.copy()
    if "y" in out.columns:
        out = out.drop(columns=["y"])
    if "duration" in out.columns:
        out = out.drop(columns=["duration"])

    if campaign_cap is None:
        campaign_cap = float(out["campaign"].quantile(0.99))

    out["campaign_capped"] = out["campaign"].clip(upper=campaign_cap)
    out["campaign_log"] = np.log1p(out["campaign"])
    out["campaign_group"] = pd.cut(
        out["campaign_capped"],
        bins=[-np.inf, 1, 2, 4, 8, np.inf],
        labels=["one", "two", "three_four", "five_eight", "nine_plus"],
    ).astype(str)
    out["age_bin"] = pd.cut(
        out["age"],
        bins=[-np.inf, 25, 35, 45, 55, 65, np.inf],
        labels=["under_25", "25_35", "35_45", "45_55", "55_65", "over_65"],
    ).astype(str)
    out["education_ord"] = education_order(out["education"])
    out["pdays_missing"] = (out["pdays"] == 999).astype(int)
    out["pdays_clean"] = np.where(out["pdays"] == 999, 0, out["pdays"])
    out["pdays_recent"] = (out["pdays"] <= 7).astype(int)
    out["contacted_before"] = ((out["pdays"] != 999) | (out["previous"] > 0)).astype(int)
    out["previous_success"] = (out["poutcome"] == "success").astype(int)
    out["previous_failure"] = (out["poutcome"] == "failure").astype(int)
    out["previous_group"] = pd.cut(
        out["previous"],
        bins=[-np.inf, 0, 1, 2, np.inf],
        labels=["none", "one", "two", "three_plus"],
    ).astype(str)
    out["emp_regime"] = pd.cut(
        out["emp.var.rate"],
        bins=[-np.inf, -2, 0, 1, np.inf],
        labels=["recession", "weak", "neutral", "expansion"],
    ).astype(str)
    out["euribor_regime"] = pd.cut(
        out["euribor3m"],
        bins=[-np.inf, 1, 2, 4, np.inf],
        labels=["very_low", "low", "medium", "high"],
    ).astype(str)
    out["confidence_regime"] = pd.cut(
        out["cons.conf.idx"],
        bins=[-np.inf, -45, -40, -35, np.inf],
        labels=["very_low", "low", "medium", "high"],
    ).astype(str)
    out["employment_regime"] = pd.cut(
        out["nr.employed"],
        bins=[-np.inf, 5050, 5150, 5200, np.inf],
        labels=["low", "medium", "high", "very_high"],
    ).astype(str)
    out["contact_month"] = out["contact"].astype(str) + "_" + out["month"].astype(str)
    out["poutcome_previous"] = out["poutcome"].astype(str) + "_" + out["previous_group"].astype(str)
    pdays_status = pd.Series(
        np.where(out["pdays_missing"] == 1, "not_contacted", "contacted"),
        index=out.index,
    )
    out["pdays_previous"] = pdays_status.astype(str) + "_" + out["previous_group"].astype(str)
    out["job_education"] = out["job"].astype(str) + "_" + out["education"].astype(str)
    out["contact_poutcome"] = out["contact"].astype(str) + "_" + out["poutcome"].astype(str)
    out["month_emp_regime"] = out["month"].astype(str) + "_" + out["emp_regime"].astype(str)
    return out, campaign_cap


class TabPreprocessor:
    def __init__(self, min_count=30):
        self.min_count = min_count

    def fit(self, df):
        frame, self.campaign_cap = add_features(df)
        self.cat_cols = frame.select_dtypes(include=["object", "category"]).columns.tolist()
        self.num_cols = [c for c in frame.columns if c not in self.cat_cols]

        self.cat_maps = {}
        self.cardinalities = []
        for col in self.cat_cols:
            values = frame[col].fillna("__missing__").astype(str)
            counts = values.value_counts()
            keep = counts[counts >= self.min_count].index.tolist()
            if not keep:
                keep = counts.index[:1].tolist()
            levels = ["__unknown__"] + sorted(keep)
            self.cat_maps[col] = {level: i for i, level in enumerate(levels)}
            self.cardinalities.append(len(levels))

        num = frame[self.num_cols].astype(float)
        self.num_mean = num.mean()
        self.num_std = num.std().replace(0, 1).fillna(1)
        return self

    def transform(self, df):
        frame, _ = add_features(df, self.campaign_cap)
        cat_arrays = []
        for col in self.cat_cols:
            mapping = self.cat_maps[col]
            values = frame[col].fillna("__missing__").astype(str)
            cat_arrays.append(values.map(mapping).fillna(0).astype("int64").to_numpy())
        x_cat = np.stack(cat_arrays, axis=1).astype("int64")
        x_num = ((frame[self.num_cols].astype(float) - self.num_mean) / self.num_std).fillna(0).to_numpy("float32")
        return x_cat, x_num


y = (train_df["y"] == "yes").astype("int64").to_numpy()
y_test = (test_df["y"] == "yes").astype("int64").to_numpy()

train_idx, valid_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

pre = TabPreprocessor(min_count=30).fit(train_df.iloc[train_idx])
train_cat, train_num = pre.transform(train_df.iloc[train_idx])
valid_cat, valid_num = pre.transform(train_df.iloc[valid_idx])
test_cat, test_num = pre.transform(test_df)
y_train = y[train_idx]
y_valid = y[valid_idx]

print("Categorical columns:", len(pre.cat_cols))
print("Numerical columns:", len(pre.num_cols))
print("Categorical cardinalities:", pre.cardinalities)

Categorical columns: 23
Numerical columns: 18
Categorical cardinalities: [13, 5, 8, 3, 4, 4, 3, 11, 6, 4, 6, 7, 5, 4, 4, 5, 5, 20, 8, 8, 73, 7, 21]


In [5]:
class BankDataset(Dataset):
    def __init__(self, x_cat, x_num, y=None):
        self.x_cat = torch.as_tensor(x_cat, dtype=torch.long)
        self.x_num = torch.as_tensor(x_num, dtype=torch.float32)
        self.y = None if y is None else torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x_cat)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x_cat[idx], self.x_num[idx]
        return self.x_cat[idx], self.x_num[idx], self.y[idx]


class FTTransformer(nn.Module):
    def __init__(self, cat_cardinalities, n_num, d_token=48, n_heads=4, n_layers=3, dropout=0.15):
        super().__init__()
        if d_token % n_heads != 0:
            raise ValueError("d_token must be divisible by n_heads")
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(cardinality, d_token) for cardinality in cat_cardinalities
        ])
        self.num_projections = nn.ModuleList([
            nn.Sequential(nn.Linear(1, d_token), nn.LayerNorm(d_token)) for _ in range(n_num)
        ])
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token, 1),
        )
        nn.init.normal_(self.cls, std=0.02)

    def forward(self, x_cat, x_num):
        tokens = []
        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat[:, i]))
        for i, proj in enumerate(self.num_projections):
            tokens.append(proj(x_num[:, i : i + 1]))
        x = torch.stack(tokens, dim=1)
        cls = self.cls.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.encoder(x)
        return self.head(x[:, 0]).squeeze(1)

In [6]:
def predict_proba(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                x_cat, x_num, _ = batch
            else:
                x_cat, x_num = batch
            x_cat = x_cat.to(DEVICE)
            x_num = x_num.to(DEVICE)
            logits = model(x_cat, x_num)
            probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)


def train_one(config):
    batch_size = config.get("batch_size", 512)
    train_loader = DataLoader(
        BankDataset(train_cat, train_num, y_train),
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda"),
    )
    valid_loader = DataLoader(
        BankDataset(valid_cat, valid_num, y_valid),
        batch_size=2048,
        shuffle=False,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda"),
    )
    model = FTTransformer(
        pre.cardinalities,
        train_num.shape[1],
        d_token=config["d_token"],
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        dropout=config["dropout"],
    ).to(DEVICE)

    pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32, device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_auc = -np.inf
    best_state = None
    stale_epochs = 0
    start = time.time()

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        running_loss = 0.0
        for x_cat, x_num, target in train_loader:
            x_cat = x_cat.to(DEVICE)
            x_num = x_num.to(DEVICE)
            target = target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits = model(x_cat, x_num)
                loss = loss_fn(logits, target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * len(target)

        valid_prob = predict_proba(model, valid_loader)
        valid_auc = roc_auc_score(y_valid, valid_prob)
        if valid_auc > best_auc:
            best_auc = valid_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1

        print(
            f"epoch {epoch:03d} | loss {running_loss / len(y_train):.4f} | valid_auc {valid_auc:.4f} | best {best_auc:.4f}"
        )
        if stale_epochs >= config["patience"]:
            break

    model.load_state_dict(best_state)
    elapsed = time.time() - start
    return model, best_auc, elapsed


CONFIGS = [
    {"name": "ft_small", "d_token": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.15, "lr": 3e-4, "weight_decay": 1e-4, "batch_size": 512, "epochs": 50, "patience": 8},
    {"name": "ft_medium", "d_token": 48, "n_heads": 4, "n_layers": 3, "dropout": 0.20, "lr": 2e-4, "weight_decay": 3e-4, "batch_size": 512, "epochs": 60, "patience": 10},
    {"name": "ft_wide", "d_token": 64, "n_heads": 8, "n_layers": 3, "dropout": 0.15, "lr": 2e-4, "weight_decay": 1e-4, "batch_size": 512, "epochs": 60, "patience": 10},
]

results = []
best = None
for config in CONFIGS:
    print("\nTraining", config["name"])
    model, best_auc, elapsed = train_one(config)
    valid_loader = DataLoader(BankDataset(valid_cat, valid_num, y_valid), batch_size=2048, shuffle=False)
    test_loader = DataLoader(BankDataset(test_cat, test_num, y_test), batch_size=2048, shuffle=False)
    valid_prob = predict_proba(model, valid_loader)
    test_prob = predict_proba(model, test_loader)
    metrics = evaluate_points(config["name"], y_test, test_prob, y_valid, valid_prob)
    metrics["valid_auc"] = best_auc
    metrics["seconds"] = elapsed
    results.append(metrics)
    if best is None or best_auc > best["valid_auc"]:
        best = {"config": config, "model": model, "valid_auc": best_auc, "valid_prob": valid_prob, "test_prob": test_prob}

transformer_metrics = pd.concat(results, ignore_index=True)
transformer_metrics.sort_values("harmonic_acc_sens_spec", ascending=False)


Training ft_small


/tmp/ipykernel_524/4087044970.py:37: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
/tmp/ipykernel_524/2317673663.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
/tmp/ipykernel_524/2317673663.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


epoch 001 | loss 1.1178 | valid_auc 0.7726 | best 0.7726


/tmp/ipykernel_524/2317673663.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


epoch 002 | loss 1.0180 | valid_auc 0.7870 | best 0.7870


...

/tmp/ipykernel_524/2317673663.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


epoch 023 | loss 0.9161 | valid_auc 0.7950 | best 0.8005


/tmp/ipykernel_524/2317673663.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


epoch 024 | loss 0.9211 | valid_auc 0.7964 | best 0.8005


,threshold,accuracy,sensitivity,specificity,balanced_accuracy,harmonic_acc_sens_spec,business_score,precision,f1,tp,tn,fp,fn,model,operating_point,auc,valid_auc,seconds
11,0.653,0.852495,0.571121,0.888220,0.729670,0.740774,0.747090,0.393467,0.465934,530,6492,817,398,ft_wide,business_threshold,0.774434,0.800517,24.601768
0,0.500,0.834163,0.589440,0.865235,0.727337,0.740542,0.742488,0.357050,0.444715,547,6324,985,381,ft_small,default_0.50,0.772943,0.797857,30.568270
10,0.535,0.827729,0.595905,0.857162,0.726534,0.740213,0.740886,0.346274,0.438020,553,6265,1044,375,ft_wide,harmonic_threshold,0.774434,0.800517,24.601768
3,0.613,0.857715,0.564655,0.894924,0.729790,0.739956,0.747933,0.405573,0.472072,524,6541,768,404,ft_small,business_threshold,0.772943,0.797857,30.568270
2,0.483,0.828457,0.593750,0.858257,0.726003,0.739567,0.740534,0.347196,0.438171,551,6273,1036,377,ft_small,harmonic_threshold,0.772943,0.797857,30.568270
8,0.500,0.818745,0.602371,0.846217,0.724294,0.738331,0.737690,0.332145,0.428188,559,6185,1124,369,ft_wide,default_0.50,0.774434,0.800517,24.601768
4,0.500,0.818502,0.601293,0.846080,0.723687,0.737690,0.737134,0.331551,0.427422,558,6184,1125,370,ft_medium,default_0.50,0.773444,0.799555,29.521504
6,0.545,0.835620,0.577586,0.868381,0.722984,0.735361,0.738959,0.357810,0.441880,536,6347,962,392,ft_medium,harmonic_threshold,0.773444,0.799555,29.521504
7,0.545,0.835620,0.577586,0.868381,0.722984,0.735361,0.738959,0.357810,0.441880,536,6347,962,392,ft_medium,business_threshold,0.773444,0.799555,29.521504
1,0.898,0.900085,0.204741,0.988371,0.596556,0.428144,0.639605,0.690909,0.315877,190,7224,85,738,ft_small,accuracy_threshold,0.772943,0.797857,30.568270


In [7]:
display_cols = [
    "model",
    "operating_point",
    "threshold",
    "accuracy",
    "sensitivity",
    "specificity",
    "balanced_accuracy",
    "harmonic_acc_sens_spec",
    "business_score",
    "auc",
    "valid_auc",
    "tp",
    "tn",
    "fp",
    "fn",
]

summary = transformer_metrics[display_cols].sort_values(
    ["harmonic_acc_sens_spec", "business_score"],
    ascending=False,
)
display(summary)
print("Best config by validation AUC:", best["config"])

,model,operating_point,threshold,accuracy,sensitivity,specificity,balanced_accuracy,harmonic_acc_sens_spec,business_score,auc,valid_auc,tp,tn,fp,fn
11,ft_wide,business_threshold,0.653,0.852495,0.571121,0.888220,0.729670,0.740774,0.747090,0.774434,0.800517,530,6492,817,398
0,ft_small,default_0.50,0.500,0.834163,0.589440,0.865235,0.727337,0.740542,0.742488,0.772943,0.797857,547,6324,985,381
10,ft_wide,harmonic_threshold,0.535,0.827729,0.595905,0.857162,0.726534,0.740213,0.740886,0.774434,0.800517,553,6265,1044,375
3,ft_small,business_threshold,0.613,0.857715,0.564655,0.894924,0.729790,0.739956,0.747933,0.772943,0.797857,524,6541,768,404
2,ft_small,harmonic_threshold,0.483,0.828457,0.593750,0.858257,0.726003,0.739567,0.740534,0.772943,0.797857,551,6273,1036,377
8,ft_wide,default_0.50,0.500,0.818745,0.602371,0.846217,0.724294,0.738331,0.737690,0.774434,0.800517,559,6185,1124,369
4,ft_medium,default_0.50,0.500,0.818502,0.601293,0.846080,0.723687,0.737690,0.737134,0.773444,0.799555,558,6184,1125,370
6,ft_medium,harmonic_threshold,0.545,0.835620,0.577586,0.868381,0.722984,0.735361,0.738959,0.773444,0.799555,536,6347,962,392
7,ft_medium,business_threshold,0.545,0.835620,0.577586,0.868381,0.722984,0.735361,0.738959,0.773444,0.799555,536,6347,962,392
1,ft_small,accuracy_threshold,0.898,0.900085,0.204741,0.988371,0.596556,0.428144,0.639605,0.772943,0.797857,190,7224,85,738


Best config by validation AUC: {'name': 'ft_wide', 'd_token': 64, 'n_heads': 8, 'n_layers': 3, 'dropout': 0.15, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 512, 'epochs': 60, 'patience': 10}


In [8]:
# Optional GPU XGBoost baseline. This is useful because tree boosting is usually very strong
# on small/medium tabular data like this dataset.
try:
    import xgboost as xgb
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    feature_train, cap = add_features(train_df.iloc[train_idx])
    feature_valid, _ = add_features(train_df.iloc[valid_idx], cap)
    feature_test, _ = add_features(test_df, cap)

    cat_cols = feature_train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in feature_train.columns if c not in cat_cols]

    pre_xgb = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), cat_cols),
            ("num", StandardScaler(), num_cols),
        ]
    )

    xgb_kwargs = dict(
        n_estimators=700,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=1,
        subsample=0.90,
        colsample_bytree=0.70,
        reg_lambda=3,
        reg_alpha=1,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=math.sqrt((y_train == 0).sum() / (y_train == 1).sum()),
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist",
    )
    if DEVICE.type == "cuda":
        xgb_kwargs["device"] = "cuda"

    xgb_model = Pipeline([
        ("prep", pre_xgb),
        ("model", xgb.XGBClassifier(**xgb_kwargs)),
    ])

    try:
        xgb_model.fit(feature_train, y_train)
    except Exception as first_error:
        print("Retrying XGBoost without device='cuda' because:", first_error)
        xgb_kwargs.pop("device", None)
        if DEVICE.type == "cuda":
            xgb_kwargs["tree_method"] = "gpu_hist"
        xgb_model = Pipeline([
            ("prep", pre_xgb),
            ("model", xgb.XGBClassifier(**xgb_kwargs)),
        ])
        xgb_model.fit(feature_train, y_train)

    valid_prob_xgb = xgb_model.predict_proba(feature_valid)[:, 1]
    test_prob_xgb = xgb_model.predict_proba(feature_test)[:, 1]
    xgb_metrics = evaluate_points("xgboost_gpu_baseline", y_test, test_prob_xgb, y_valid, valid_prob_xgb)
    combined = pd.concat([transformer_metrics, xgb_metrics], ignore_index=True)
    display(combined[display_cols[:-1] + ["tp", "tn", "fp", "fn"]].sort_values("harmonic_acc_sens_spec", ascending=False))
except Exception as exc:
    print("Skipped optional XGBoost baseline:", repr(exc))

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [21:33:32] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,model,operating_point,threshold,accuracy,sensitivity,specificity,balanced_accuracy,harmonic_acc_sens_spec,business_score,auc,valid_auc,tp,tn,fp,tp,tn,fp,fn
11,ft_wide,business_threshold,0.653,0.852495,0.571121,0.888220,0.729670,0.740774,0.747090,0.774434,0.800517,530,6492,817,530,6492,817,398
0,ft_small,default_0.50,0.500,0.834163,0.589440,0.865235,0.727337,0.740542,0.742488,0.772943,0.797857,547,6324,985,547,6324,985,381
10,ft_wide,harmonic_threshold,0.535,0.827729,0.595905,0.857162,0.726534,0.740213,0.740886,0.774434,0.800517,553,6265,1044,553,6265,1044,375
3,ft_small,business_threshold,0.613,0.857715,0.564655,0.894924,0.729790,0.739956,0.747933,0.772943,0.797857,524,6541,768,524,6541,768,404
2,ft_small,harmonic_threshold,0.483,0.828457,0.593750,0.858257,0.726003,0.739567,0.740534,0.772943,0.797857,551,6273,1036,551,6273,1036,377
8,ft_wide,default_0.50,0.500,0.818745,0.602371,0.846217,0.724294,0.738331,0.737690,0.774434,0.800517,559,6185,1124,559,6185,1124,369
15,xgboost_gpu_baseline,business_threshold,0.298,0.842176,0.575431,0.876043,0.725737,0.737694,0.742251,0.774333,NaN,534,6403,906,534,6403,906,394
14,xgboost_gpu_baseline,harmonic_threshold,0.298,0.842176,0.575431,0.876043,0.725737,0.737694,0.742251,0.774333,NaN,534,6403,906,534,6403,906,394
4,ft_medium,default_0.50,0.500,0.818502,0.601293,0.846080,0.723687,0.737690,0.737134,0.773444,0.799555,558,6184,1125,558,6184,1125,370
7,ft_medium,business_threshold,0.545,0.835620,0.577586,0.868381,0.722984,0.735361,0.738959,0.773444,0.799555,536,6347,962,536,6347,962,392


In [9]:
output_dir = Path("/content/transformer_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

best_model_path = output_dir / "ft_transformer_bank_model.pt"
torch.save(
    {
        "state_dict": best["model"].state_dict(),
        "config": best["config"],
        "cat_cols": pre.cat_cols,
        "num_cols": pre.num_cols,
        "cat_maps": pre.cat_maps,
        "cardinalities": pre.cardinalities,
        "campaign_cap": pre.campaign_cap,
        "num_mean": pre.num_mean.to_dict(),
        "num_std": pre.num_std.to_dict(),
    },
    best_model_path,
)

transformer_metrics.to_csv(output_dir / "transformer_test_metrics.csv", index=False)
pd.DataFrame(
    {
        "row_id": np.arange(len(y_test)) + 1,
        "actual": np.where(y_test == 1, "yes", "no"),
        "probability_yes": best["test_prob"],
    }
).to_csv(output_dir / "transformer_test_predictions.csv", index=False)

print("Saved:")
print(best_model_path)
print(output_dir / "transformer_test_metrics.csv")
print(output_dir / "transformer_test_predictions.csv")

Saved:
/content/transformer_outputs/ft_transformer_bank_model.pt
/content/transformer_outputs/transformer_test_metrics.csv
/content/transformer_outputs/transformer_test_predictions.csv


## How to read the result

Use the `harmonic_threshold` row if you want a balanced tradeoff across accuracy, sensitivity, and specificity.

Use the `accuracy_threshold` row only if your main goal is high raw accuracy. On this imbalanced dataset, that often reduces sensitivity sharply.

If the optional XGBoost baseline still beats the transformer, that is a valid result: tree boosting is usually very strong for small tabular datasets.